# 12. Show image detections

Run inference and draw detected boxes on the image

Before running: Set MODEL_ID and IMAGE_PATH; object detection models may return boxes

Fill in `API_URL` and `API_KEY` in the next cell; set other values if needed

In [ ]:
import requests

API_URL = ""  # Example: https://your-server/api
API_KEY = ""
VERIFY_SSL = False # Set to True if you want to verify SSL certificates
MODEL_ID = '' # The model_id can be found in the URL for that model in the MVI UI
IMAGE_PATH = 'example.jpg'

# Use the API key in each request
headers = {"X-Auth-Token": API_KEY}

from PIL import Image, ImageDraw
from IPython.display import display

with open(IMAGE_PATH, "rb") as image_file:
    files = {"files": ("image", image_file)}
    response = requests.post(f"{API_URL.rstrip('/')}/dlapis/{MODEL_ID}", headers=headers, files=files, verify=VERIFY_SSL, timeout=120)
response.raise_for_status()
result = response.json()

# Models may return classification results without boxes
image = Image.open(IMAGE_PATH).convert("RGB")
draw = ImageDraw.Draw(image)
for item in result.get("classified", []):
    print(item.get("label"), item.get("confidence"))
    if all(key in item for key in ("xmin", "ymin", "xmax", "ymax")):
        box = [item["xmin"], item["ymin"], item["xmax"], item["ymax"]]
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], box[1]), str(item.get("label", "")), fill="red")
display(image)
